In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models
import torchvision.transforms as T
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
from sklearn.preprocessing import MultiLabelBinarizer
import matplotlib.pyplot as plt
from pathlib import Path
import os
import glob
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}\n")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ========== CONFIG ==========
BASE_DIR = r"D:\Projects\CLARITY\Model\Dataset\archive"
CSV_PATH = r"D:\Projects\CLARITY\Model\Dataset\archive\Data_Entry_2017.csv"
BATCH_SIZE = 32
IMAGE_SIZE = 224

# ========== CELL 1: SETUP ==========
print("="*70)
print("[STEP 1] Environment Setup")
print("="*70 + "\n")

os.makedirs('./outputs', exist_ok=True)

# ========== CELL 2: LOAD DATASET (AS YOU PROVIDED) ==========
print("\n[STEP 2] Loading and preparing dataset...")

df = pd.read_csv(CSV_PATH)
print(f"✅ CSV loaded: {df.shape}")
print(df.head(3))

df = df[['Image Index', 'Finding Labels', 'Patient ID']].copy()

df['Finding Labels'] = df['Finding Labels'].apply(lambda x: x.split('|'))

all_labels = sorted(set([l for sublist in df['Finding Labels'] for l in sublist]))

print(f"\n✅ Total unique disease labels: {len(all_labels)}")
print(f"Labels: {all_labels}")

mlb = MultiLabelBinarizer(classes=all_labels)
label_matrix = mlb.fit_transform(df['Finding Labels'])
label_cols = mlb.classes_

df = pd.concat([df, pd.DataFrame(label_matrix, columns=label_cols)], axis=1)

print(f"\n✅ Final dataframe shape: {df.shape}")
print(f"Label columns ({len(label_cols)}): {list(label_cols)}")

print("\n[Building image path cache with glob...]")

img_lookup = {}
pattern = os.path.join(BASE_DIR, "images_*", "images", "*.png")
print(f"   Glob pattern: {pattern}")

all_images = glob.glob(pattern)
print(f"   Found {len(all_images)} images via glob")

for img_path in all_images:
    fname = os.path.basename(img_path)
    img_lookup[fname] = img_path

print(f"✅ Image cache built: {len(img_lookup)} images indexed")

print("\n[Mapping image paths to dataframe records...]")

img_paths = []
missing_count = 0

for idx_name in df['Image Index']:
    if idx_name in img_lookup:
        img_paths.append(img_lookup[idx_name])
    else:
        img_paths.append(None)
        missing_count += 1

df['img_path'] = img_paths

print(f"✅ Image paths assigned")
print(f"   Total records: {len(df)}")
print(f"   With valid paths: {len(df) - missing_count}")
print(f"   Missing paths: {missing_count}")

df_before = len(df)
df = df[df['img_path'].notna()].reset_index(drop=True)
df_after = len(df)

print(f"\n✅ After removing missing images:")
print(f"   Removed: {df_before - df_after} records")
print(f"   Remaining: {df_after} samples")

print(f"\n✅ Dataset prepared successfully!")
print(f"   Shape: {df.shape}")
print(f"   Patients: {df['Patient ID'].nunique()}")
print(f"   Label distribution (first 8):")

for i, label in enumerate(label_cols[:8]):
    count = df[label].sum()
    pct = 100 * count / len(df)
    print(f"      {i+1:2d}. {label:20s}: {count:6.0f} ({pct:5.1f}%)")

print(f"\n[Sample path verification]")
sample_paths = df['img_path'].sample(min(5, len(df)), random_state=SEED)
for i, path in enumerate(sample_paths.values, 1):
    exists = os.path.exists(path)
    size_kb = os.path.getsize(path) / 1024 if exists else 0
    status = "✅" if exists else "❌"
    print(f"   {status} Sample {i}: {os.path.basename(path)} ({size_kb:.1f} KB)")

print("\n" + "="*70)

# ========== CELL 3: CREATE DATASET CLASS ==========
print("\n[STEP 3] Creating PyTorch Dataset...")

class ChestXRayDataset(Dataset):
    def __init__(self, dataframe, label_columns, transform=None):
        self.df = dataframe
        self.label_cols = label_columns
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['img_path']
        
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        
        labels = torch.tensor([row[col] for col in self.label_cols], dtype=torch.float32)
        
        return img, labels

transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_dataset = ChestXRayDataset(df, label_cols, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"✅ Dataset created: {len(test_dataset)} samples")
print(f"✅ DataLoader created: {len(test_loader)} batches (batch_size={BATCH_SIZE})\n")

# ========== CELL 4: LOAD MODELS ==========
print("="*70)
print("[STEP 4] Loading Models")
print("="*70 + "\n")

NUM_CLASSES = len(label_cols)

print("[Loading DenseNet121...]")
densenet = models.densenet121(weights=None)
densenet.classifier = nn.Linear(densenet.classifier.in_features, NUM_CLASSES)
densenet.load_state_dict(torch.load('./Model 6 (more epochs)/outputs_best_model/best_densenet121_auc_0.9170.pth', map_location=DEVICE))
densenet.eval().to(DEVICE)
print(f"✅ DenseNet121 loaded on {DEVICE}")
print(f"   Total parameters: {sum(p.numel() for p in densenet.parameters()):,}\n")

print("[Loading ResNet152...]")
resnet = models.resnet152(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, NUM_CLASSES)
resnet.load_state_dict(torch.load('./Model 7 (more epochs)/outputs_resnet152_15epochs/best_resnet152_auc_0.9167.pth', map_location=DEVICE))
resnet.eval().to(DEVICE)
print(f"✅ ResNet152 loaded on {DEVICE}")
print(f"   Total parameters: {sum(p.numel() for p in resnet.parameters()):,}\n")

# ========== CELL 5: INFERENCE ==========
print("="*70)
print("[STEP 5] Running Inference")
print("="*70 + "\n")

def predict_model(model, dataloader, model_name):
    all_labels = []
    all_probs = []
    print(f"   Running inference...")
    with torch.no_grad():
        for i, (x, y) in enumerate(dataloader):
            x = x.to(DEVICE)
            logits = model(x)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(y.numpy())
            if (i+1) % 20 == 0:
                print(f"   Processed {(i+1)*BATCH_SIZE}/{len(test_dataset)} samples...")
    return np.vstack(all_probs), np.vstack(all_labels)

print("[DenseNet121 Inference...]")
probs_densenet, labels = predict_model(densenet, test_loader, "DenseNet121")
print(f"✅ DenseNet121 predictions: shape={probs_densenet.shape}\n")

print("[ResNet152 Inference...]")
probs_resnet, _ = predict_model(resnet, test_loader, "ResNet152")
print(f"✅ ResNet152 predictions: shape={probs_resnet.shape}\n")

# ========== CELL 6: COMPUTE METRICS ==========
print("="*70)
print("[STEP 6] Computing Metrics")
print("="*70 + "\n")

def get_classwise_metrics(y_true, y_pred, threshold=0.5):
    y_pred_bin = (y_pred >= threshold).astype(int)
    metrics = {"f1": [], "auc": [], "precision": [], "recall": []}
    for i in range(y_true.shape[1]):
        try:
            metrics["f1"].append(f1_score(y_true[:, i], y_pred_bin[:, i], zero_division=0))
            metrics["auc"].append(roc_auc_score(y_true[:, i], y_pred[:, i]))
            metrics["precision"].append(precision_score(y_true[:, i], y_pred_bin[:, i], zero_division=0))
            metrics["recall"].append(recall_score(y_true[:, i], y_pred_bin[:, i], zero_division=0))
        except Exception as e:
            metrics["f1"].append(0)
            metrics["auc"].append(0)
            metrics["precision"].append(0)
            metrics["recall"].append(0)
    return metrics

print("[Computing metrics...]")
metrics_densenet = get_classwise_metrics(labels, probs_densenet, threshold=0.5)
metrics_resnet = get_classwise_metrics(labels, probs_resnet, threshold=0.5)
print("✅ Metrics computed\n")

# ========== CELL 7: PRINT SUMMARY ==========
print("="*70)
print("AVERAGE METRICS COMPARISON")
print("="*70)
print(f"{'Metric':<12} {'DenseNet121':<20} {'ResNet152':<20}")
print("-"*70)
for metric in ["f1", "auc", "precision", "recall"]:
    dn_avg = np.mean(metrics_densenet[metric])
    rn_avg = np.mean(metrics_resnet[metric])
    print(f"{metric.upper():<12} {dn_avg:<20.4f} {rn_avg:<20.4f}")
print("="*70 + "\n")

# ========== CELL 8: PLOT ==========
print("[Creating Plots...]")
x = np.arange(NUM_CLASSES)
bar_width = 0.35

fig, axs = plt.subplots(2, 2, figsize=(20, 12))

metrics_list = ["f1", "auc", "precision", "recall"]
for idx, metric in enumerate(metrics_list):
    ax = axs[idx//2, idx%2]
    
    ax.bar(x - bar_width/2, metrics_densenet[metric], bar_width, label="DenseNet121", alpha=0.8)
    ax.bar(x + bar_width/2, metrics_resnet[metric], bar_width, label="ResNet152", alpha=0.8)
    
    ax.set_xlabel("Disease Class", fontsize=12, fontweight='bold')
    ax.set_ylabel(metric.upper(), fontsize=12, fontweight='bold')
    ax.set_title(f"Class-wise {metric.upper()} Comparison", fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(label_cols, rotation=45, ha='right')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('./outputs/model_comparison_metrics.png', dpi=300, bbox_inches='tight')
print("✅ Plot saved: ./outputs/model_comparison_metrics.png\n")
plt.show()

# ========== CELL 9: SAVE METRICS ==========
print("[Saving detailed metrics to CSV...]")
df_metrics = pd.DataFrame({
    'Class': label_cols,
    'DenseNet121_F1': metrics_densenet['f1'],
    'ResNet152_F1': metrics_resnet['f1'],
    'DenseNet121_AUC': metrics_densenet['auc'],
    'ResNet152_AUC': metrics_resnet['auc'],
    'DenseNet121_Precision': metrics_densenet['precision'],
    'ResNet152_Precision': metrics_resnet['precision'],
    'DenseNet121_Recall': metrics_densenet['recall'],
    'ResNet152_Recall': metrics_resnet['recall'],
})

df_metrics.to_csv('./outputs/model_comparison_metrics.csv', index=False)
print("✅ Metrics saved: ./outputs/model_comparison_metrics.csv\n")

print(df_metrics.to_string())

print("\n" + "="*70)
print("✅ COMPLETE - All plots and metrics generated!")
print("="*70)

Using device: cuda

[STEP 1] Environment Setup


[STEP 2] Loading and preparing dataset...
✅ CSV loaded: (112120, 12)
        Image Index          Finding Labels  Follow-up #  Patient ID  \
0  00000001_000.png            Cardiomegaly            0           1   
1  00000001_001.png  Cardiomegaly|Emphysema            1           1   
2  00000001_002.png   Cardiomegaly|Effusion            2           1   

   Patient Age Patient Gender View Position  OriginalImage[Width  Height]  \
0           58              M            PA                 2682     2749   
1           58              M            PA                 2894     2729   
2           58              M            PA                 2500     2048   

   OriginalImagePixelSpacing[x     y]  Unnamed: 11  
0                        0.143  0.143          NaN  
1                        0.143  0.143          NaN  
2                        0.168  0.168          NaN  

✅ Total unique disease labels: 15
Labels: ['Atelectasis', 'Cardiomegaly